# Two-sequence LoRA overfitting diagnostic

Use **AppliedTaskByte** and close other model kernels to release GPU memory. This notebook starts from your local pretrained Qwen weights, selects two training sequences with multiple speakers, and trains the same decoder-only QLoRA and assistant-answer loss used in `train_VL.ipynb`.

It measures predictions on those same examples before training and every 20 optimizer updates. This is a memorization test, not validation performance. It intentionally does not change loss masking, character numbering, image resolution, or model components.

Outputs are isolated under `outputs/vl_training/overfit_two_sequences_v1/`. Main training checkpoints and validation predictions are untouched. Cells are left unexecuted.


## 1. Settings

100 optimizer updates repeatedly revisit two examples. At each check, report both exact reference numbering and accuracy after sequence-wide character renaming. Set a new output folder for another experiment.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib.metadata
import importlib.util
import json
import re
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'TaskData').is_dir() or (p / 'manga_notebook_codebase/TaskData').is_dir())
if not (ROOT / 'TaskData').is_dir():
    ROOT = ROOT / 'manga_notebook_codebase'
ROOT = ROOT.resolve()
MODEL_PATH = Path(r'C:\MODELS\HF\Qwen2.5-VL-3B-Instruct')
PROMPT_DIR = ROOT / 'outputs/speaker_prompts/train'
PREP_DIR = ROOT / 'outputs/transcript_prep_vl/train'
OUTPUT_DIR = ROOT / 'outputs/vl_training/overfit_two_sequences_lines_v1'

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28
MAX_SEQUENCE_TOKENS = 6144
MAX_NEW_TOKENS = 2048
LORA_RANK = 16
LEARNING_RATE = 1e-4
MAX_STEPS = 100
CHECK_EVERY = 20
SEED = 42
SEQUENCE_IDS = None  # Or supply exactly two training sequence IDs.

import sys
import importlib
sys.path.insert(0, str(ROOT / 'src'))
import vl_prompts
importlib.reload(vl_prompts)  # Pick up edits even in an existing kernel.
from vl_prompts import (OUTPUT_FORMAT, ordered_groups, build_speaker_prompt,
                        build_speaker_target, parse_speaker_output, permute_speaker_labels)


## 2. Check data and select two sequences

Selection uses training records only. Each sequence must have at least eight supervised groups and at least two characters appearing twice each. Choose shorter eligible sequences deterministically to make the test smaller. No target is rewritten.


In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()
            if line.strip()]


def file_hash(path):
    # Read large model files in chunks to keep memory use small.
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


examples_path = PROMPT_DIR / 'training_examples.jsonl'
if not examples_path.is_file():
    raise FileNotFoundError('Prepare training transcripts and prompts first; see the first cell.')
summary = json.loads((PROMPT_DIR / 'summary.json').read_text(encoding='utf-8'))
if summary.get('output_format') != OUTPUT_FORMAT:
    raise ValueError('Rerun build_speaker_prompt_Vl.ipynb for ordered speaker targets.')
if summary['split'] != 'train':
    raise ValueError('Expected training prompt outputs')
if file_hash(examples_path) != summary['output_sha256']['training_examples.jsonl']:
    raise ValueError('Training examples changed; regenerate prompts.')
for name, key in [('inputs.jsonl', 'inputs'), ('targets.jsonl', 'targets')]:
    if file_hash(PREP_DIR / name) != summary['source_sha256'][key]:
        raise ValueError('Training preparation changed; regenerate prompts.')
split_path = ROOT / 'outputs/split.json'
if file_hash(split_path) != summary['source_sha256']['split']:
    raise ValueError('Saved split changed; regenerate prompts.')
if file_hash(ROOT / 'src/vl_prompts.py') != summary['shared_prompt_sha256']:
    raise ValueError('Shared prompt code changed; regenerate prompts.')

examples = read_jsonl(examples_path)
saved_split = json.loads(split_path.read_text(encoding='utf-8'))
ids = [example['sequence_id'] for example in examples]
if not ids or len(ids) != len(set(ids)):
    raise ValueError('Training examples must be nonempty with unique sequence IDs')
if not set(ids).issubset(set(saved_split['train'])) or set(ids) & set(saved_split['val']):
    raise ValueError('Training examples contain sequences outside the saved training split')

# Check images and answer IDs while no GPU memory has been allocated.
image_hashes = {}
for example in examples:
    if len(example['images']) != 3:
        raise ValueError('Each training example needs three images')
    # Parse using the saved ordered groups, never a set or alphabetical ID order.
    requested = example['requested_groups']
    answer = parse_speaker_output(example['assistant_response'], requested)
    if not answer or example['prompt'] != build_speaker_prompt(requested):
        raise ValueError('Training prompt or target does not match requested groups')
    if example.get('output_format') != OUTPUT_FORMAT:
        raise ValueError('Old training target format; regenerate prompts')
    for image in example['images']:
        path = Path(image).resolve()
        if not path.is_relative_to(PREP_DIR.resolve()) or not path.is_file():
            raise ValueError(f'Missing or unsafe training image: {path}')
        relative = str(path.relative_to(PREP_DIR.resolve()))
        image_hashes[relative] = file_hash(path)
        if image_hashes[relative] != summary['annotated_image_sha256'][relative]:
            raise ValueError(f'Training image changed: {path}')
print('Training sequences:', len(examples))
print('Supervised groups:', sum(len(e['requested_groups']) for e in examples))


In [ ]:
available = {example['sequence_id']: example for example in examples}


def suitable_example(example):
    counts = Counter(parse_speaker_output(example['assistant_response'], example['requested_groups']).values())
    repeated_characters = [speaker for speaker, count in counts.items()
                           if speaker.startswith('char') and count >= 2]
    return sum(counts.values()) >= 8 and len(repeated_characters) >= 2


eligible = sorted([example for example in examples if suitable_example(example)],
                  key=lambda e: (len(e['requested_groups']), e['sequence_id']))
if SEQUENCE_IDS is None:
    examples = eligible[:2]
else:
    if len(SEQUENCE_IDS) != 2 or len(set(SEQUENCE_IDS)) != 2:
        raise ValueError('Supply exactly two distinct training sequence IDs')
    examples = [available[sid] for sid in SEQUENCE_IDS]
    if not all(suitable_example(example) for example in examples):
        raise ValueError('Each chosen sequence must contain multiple repeated speakers')
if len(examples) != 2:
    raise ValueError('Need two eligible training sequences')
for example in examples:
    print(example['sequence_id'], dict(Counter(parse_speaker_output(example['assistant_response'], example['requested_groups']).values())))

# A constant speaker is a baseline to beat, not evidence of learning attribution.
majority_correct = sum(max(Counter(speaker for speaker in parse_speaker_output(e['assistant_response'], e['requested_groups']).values()
                                  if speaker not in {'UNKNOWN', 'NARRATION'}).values()) for e in examples)
total_targets = sum(len(e['requested_groups']) for e in examples)
print('Single-character baseline:', round(majority_correct / total_targets, 4))


## 3. Load a fresh base model and decoder LoRA

This uses the same NF4 base, rank 16 adapters, and frozen vision encoder/merger as main training. No existing adapter is loaded.


In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from transformers import Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError('Use the CUDA-enabled AppliedTaskByte kernel')
set_seed(SEED)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(
    MODEL_PATH, local_files_only=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'right'

# The base weights are quantized and remain frozen.
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH, local_files_only=True, dtype=compute_dtype,
    quantization_config=quantization, device_map={'': 0}, attn_implementation='sdpa')
model = prepare_model_for_kbit_training(
    model, gradient_checkpointing_kwargs={'use_reentrant': False})
model.config.use_cache = False  # Training uses checkpointing instead of generation caching.

# Train attention and MLP adapters in the language decoder only.
suffixes = {'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'}
target_modules = [name for name, _ in model.named_modules()
                  if '.language_model.layers.' in name and name.split('.')[-1] in suffixes]
if not target_modules:
    raise ValueError('No decoder LoRA modules found; inspect the model architecture')
lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.05,
    target_modules=target_modules, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
for name, parameter in model.named_parameters():
    if parameter.requires_grad and ('lora_' not in name or '.language_model.layers.' not in name):
        raise ValueError(f'Unexpected trainable parameter: {name}')
model.print_trainable_parameters()


## 4. Build batches with speaker-line loss

Mask user/image tokens and supervise only speaker lines plus the chat ending. Apply one seeded character-label permutation per presentation; keep UNKNOWN and NARRATION fixed. The diagnostic scores character partitions after permutation matching, since exact numbering is arbitrary. Random targets still have ambiguity under exact-token loss.


In [ ]:
from PIL import Image
from qwen_vl_utils import process_vision_info


def collate_example(records):
    if len(records) != 1:
        raise ValueError('This simple collator supports batch size 1')
    example = records[0]
    content = []
    for path in example['images']:
        with Image.open(path) as source:
            image = source.convert('RGB')
        content.append(dict(type='image', image=image,
                            min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS))
    content.append(dict(type='text', text=example['prompt']))
    user_message = dict(role='user', content=content)
    # Shuffle identities once for this entire three-page presentation.
    # Trainer seeds and checkpoints Python's RNG; workers remain zero for replay.
    speakers = list(parse_speaker_output(
        example['assistant_response'], example['requested_groups']).values())
    shuffled = permute_speaker_labels(speakers)
    assistant_text = '\n'.join(shuffled)
    assistant_message = dict(role='assistant', content=assistant_text)

    # The generation prompt is the exact prefix before the assistant answer.
    prompt_text = processor.apply_chat_template(
        [user_message], tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(
        [user_message, assistant_message], tokenize=False, add_generation_prompt=False)
    images, videos = process_vision_info([user_message])
    try:
        prefix = processor(text=[prompt_text], images=images, videos=videos, return_tensors='pt')
        batch = processor(text=[full_text], images=images, videos=videos, return_tensors='pt')
    finally:
        for item in content:
            if item['type'] == 'image':
                item['image'].close()
        for image in images:
            image.close()
    prefix_length = prefix['input_ids'].shape[1]
    if not torch.equal(batch['input_ids'][:, :prefix_length], prefix['input_ids']):
        raise ValueError('Chat-template token prefix differs; refusing incorrect loss masking')
    if batch['input_ids'].shape[1] > MAX_SEQUENCE_TOKENS:
        raise ValueError(f"{example['sequence_id']}: sequence exceeds MAX_SEQUENCE_TOKENS")
    labels = batch['input_ids'].clone()
    labels[:, :prefix_length] = -100  # Ignore all user text and image tokens; supervise speaker lines only.
    labels[batch['attention_mask'] == 0] = -100
    if not (labels != -100).any():
        raise ValueError('No assistant tokens remain for training')
    batch['labels'] = labels
    return batch


# Inspect token counts before starting optimization; this does not train the model.
example_batch = collate_example([examples[0]])
print('Total tokens:', example_batch['input_ids'].shape[1])
print('Supervised answer tokens:', int((example_batch['labels'] != -100).sum()))
print('Example supervised lines:', processor.decode(
    example_batch['labels'][example_batch['labels'] != -100], skip_special_tokens=True))
del example_batch


## 5. Generate and score the two examples

Generation receives images and the user prompt only. Strictly require one valid speaker line per requested group. Malformed responses receive no accepted assignments. Match character identities one-to-one across all three pages; UNKNOWN and NARRATION stay fixed.


In [ ]:
# Reuse only the official character-matching helper.
spec = importlib.util.spec_from_file_location('manga_scorer', ROOT / 'TaskData/score.py')
scorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer)


def predict_example(example):
    content = []
    for path in example['images']:
        with Image.open(path) as source:
            image = source.convert('RGB')
        content.append(dict(type='image', image=image,
                            min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS))
    content.append(dict(type='text', text=example['prompt']))
    messages = [dict(role='user', content=content)]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    try:
        batch = processor(text=[text], images=images, videos=videos,
                          return_tensors='pt').to('cuda')
    finally:
        for item in content:
            if item['type'] == 'image':
                item['image'].close()
        for image in images:
            image.close()
    # Teacher answers are not part of this generation call.
    with torch.inference_mode():
        generated = model.generate(**batch, do_sample=False, num_beams=1,
                                   use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
    answer_tokens = generated[0, batch['input_ids'].shape[1]:]
    raw = processor.decode(answer_tokens, skip_special_tokens=True,
                           clean_up_tokenization_spaces=False)
    return raw, int(answer_tokens.numel()) >= MAX_NEW_TOKENS


def score_response(raw, reference):
    try:
        requested = [dict(group_id=gid) for gid in reference]
        valid = parse_speaker_output(raw, requested)
        extras, invalid, parse_error = [], [], None
    except (ValueError, TypeError) as error:
        valid, extras, invalid = {}, [], []
        parse_error = str(error)
    fixed = {'UNKNOWN', 'NARRATION'}
    pairs = [(target, valid[gid], 1.0) for gid, target in reference.items()
             if gid in valid and target not in fixed and valid[gid] not in fixed]
    mapping = scorer.best_mapping(pairs)
    mapping.update(UNKNOWN='UNKNOWN', NARRATION='NARRATION')
    missing = [gid for gid in reference if gid not in valid]
    details = [dict(group_id=gid, target=target, prediction=valid.get(gid),
                    mapped_prediction=mapping.get(valid.get(gid)),
                    correct=mapping.get(valid.get(gid)) == target)
               for gid, target in reference.items()]
    return dict(correct=sum(group['correct'] for group in details), total=len(reference),
                exact_correct=sum(valid.get(gid) == target for gid, target in reference.items()),
                label_mapping=mapping, missing_ids=missing, extra_ids=extras,
                invalid_speaker_ids=invalid, parse_error=parse_error,
                predicted_label_counts=dict(Counter(valid.values())), groups=details)


history = []


def check_predictions(step):
    # Evaluate temporarily, then restore the model's previous training state.
    was_training = model.training
    model.eval()
    results = []
    try:
        for example in examples:
            raw, hit_limit = predict_example(example)
            reference = parse_speaker_output(example['assistant_response'], example['requested_groups'])
            results.append(dict(sequence_id=example['sequence_id'], raw_response=raw,
                                hit_token_limit=hit_limit, **score_response(raw, reference)))
    finally:
        model.train(was_training)
    total = sum(row['total'] for row in results)
    complete = all(not (row['missing_ids'] or row['extra_ids'] or row['invalid_speaker_ids']
                        or row['parse_error'] or row['hit_token_limit']) for row in results)
    record = dict(step=step, mapped_accuracy=sum(row['correct'] for row in results) / total,
                  exact_accuracy=sum(row['exact_correct'] for row in results) / total,
                  complete_responses=complete, results=results)
    history.append(record)
    # Save every check, including raw answers and per-group errors.
    (OUTPUT_DIR / 'checks.json').write_text(json.dumps(history, indent=2) + '\n', encoding='utf-8')
    print('Step', step, 'mapped accuracy', round(record['mapped_accuracy'], 4),
          'exact accuracy', round(record['exact_accuracy'], 4), flush=True)
    for row in results:
        print(row['sequence_id'], row['predicted_label_counts'], flush=True)
    return record


## 6. Train, checking every 20 updates

Save checkpoints every 20 updates, retaining two. Stop early only when both sequences have complete responses and 100% accuracy with a consistent character renaming. Exact-number accuracy is also recorded. Otherwise continue to the configured limit.


In [ ]:
from transformers import TrainerCallback

if CHECK_EVERY < 1 or MAX_STEPS < 1:
    raise ValueError('Step settings must be positive')
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise ValueError('Diagnostic output already exists; use a new OUTPUT_DIR')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Record the selected examples and settings, including the source code/runtime.
notebook = json.loads((ROOT / 'notebooks/overfit_VL.ipynb').read_text(encoding='utf-8'))
settings = dict(sequence_ids=[e['sequence_id'] for e in examples], seed=SEED,
                max_steps=MAX_STEPS, check_every=CHECK_EVERY, learning_rate=LEARNING_RATE,
                rank=LORA_RANK, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
                max_sequence_tokens=MAX_SEQUENCE_TOKENS, max_new_tokens=MAX_NEW_TOKENS,
                quantization='NF4', compute_dtype=str(compute_dtype),
                loss='Assistant speaker-line token loss', output_format=OUTPUT_FORMAT,
                label_permutation='per-presentation Python RNG', target_modules=target_modules,
                source_examples_sha256=file_hash(examples_path), image_sha256=image_hashes,
                model_path=str(MODEL_PATH),
                model_sha256={p.name: file_hash(p) for p in MODEL_PATH.iterdir()
                              if p.is_file() and p.suffix in {'.json', '.safetensors', '.txt'}},
                notebook_code_sha256=hashlib.sha256(json.dumps([
                    c['source'] for c in notebook['cells'] if c['cell_type'] == 'code']).encode()).hexdigest(),
                packages={name: importlib.metadata.version(name) for name in
                          ['torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes', 'qwen-vl-utils']})
(OUTPUT_DIR / 'settings.json').write_text(json.dumps(settings, indent=2) + '\n', encoding='utf-8')
(OUTPUT_DIR / 'selected_examples.jsonl').write_text(
    ''.join(json.dumps(e, ensure_ascii=False) + '\n' for e in examples), encoding='utf-8')

# Establish the baseline on exactly the same two examples before optimization.
check_predictions(0)


class CheckMemorization(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        check = check_predictions(state.global_step)
        if check['complete_responses'] and check['mapped_accuracy'] == 1.0:
            control.should_training_stop = True
            print('Both sequences memorized; stopping the diagnostic.')
        return control


arguments = TrainingArguments(
    output_dir=str(OUTPUT_DIR), max_steps=MAX_STEPS,
    per_device_train_batch_size=1, gradient_accumulation_steps=1,
    learning_rate=LEARNING_RATE, warmup_steps=5, lr_scheduler_type='constant_with_warmup',
    optim='adamw_torch', max_grad_norm=1.0,
    bf16=compute_dtype == torch.bfloat16, fp16=compute_dtype == torch.float16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=1, save_strategy='steps', save_steps=CHECK_EVERY, save_total_limit=2,
    eval_strategy='no', remove_unused_columns=False, dataloader_num_workers=0,
    report_to='none', seed=SEED, data_seed=SEED)
trainer = Trainer(model=model, args=arguments, train_dataset=examples,
                  data_collator=collate_example, processing_class=processor,
                  callbacks=[CheckMemorization()])
result = trainer.train()
model.save_pretrained(OUTPUT_DIR / 'adapter')
processor.save_pretrained(OUTPUT_DIR / 'adapter')
trainer.save_state()
trainer.save_metrics('train', result.metrics)
final = check_predictions(trainer.state.global_step)
print('Final diagnostic accuracy:', final['mapped_accuracy'])


## 7. Read the result

- Near-perfect prediction on both examples: the setup can memorize the requested assignments. Next check held-out generalization; this does not prove visual grounding.
- Low training loss but constant labels: the original objective is still not producing speaker discrimination. Inspect speaker-value loss, IDs, and the resized images next.
- Cannot fit either example: inspect data/targets, masking, gradients, and image processing before a larger run.

A failed diagnostic does not by itself prove that the vision encoder needs training. Review per-group errors and whether the same consistent label renaming explains the mismatch.


In [ ]:
print('Checks:')
for check in history:
    print(check['step'], 'mapped:', round(check['mapped_accuracy'], 4),
          'exact:', round(check['exact_accuracy'], 4))
print('Final errors:')
for row in history[-1]['results']:
    print(row['sequence_id'])
    print(json.dumps([group for group in row['groups'] if not group['correct']], indent=2))
print('Saved reports:', OUTPUT_DIR / 'checks.json')
